# Exercises XP: Day 2
Follow the instructions. Where you see TODO, add your answer before running/continuing.


## What You'll Learn
- Deepen your understanding of core LLM concepts.
- Apply theory to practical scenarios.
- Develop critical thinking on LLM applications and ethics.
- Compare/contrast transformer architectures and techniques.


## What You Will Create
- Comparative tables (NLP paradigms and BERT variants)
- Architecture/application write-ups
- Pretraining benefits and ethical considerations
- Analyses on attention and positional encoding
- Model selection justifications across tasks
- Notes on softmax temperature and its effects
- Scenario-based answers applying learned concepts


## 🌟 Exercise 1: Traditional vs. Modern NLP: Comparative Analysis
1) Complete the table (replace each TODO):

| Aspect | Traditional NLP | Modern NLP |
|---|---|---|
| Feature Engineering | Manual (TF-IDF, n-grams, POS tags, hand-built lexicons) | Learned automatically by the network from raw/tokenized text |
| Word Representations | Discrete, sparse vectors (one-hot, bag-of-words) — no notion of similarity | Dense embeddings; contextual embeddings (BERT/GPT) where meaning shifts with context |
| Model Architectures | Naive Bayes, HMMs, CRFs, SVMs, rule-based grammars, early RNNs | Transformers (self-attention), encoder/decoder/encoder-decoder LLMs |
| Training Methodology | Task-specific supervised training from scratch on labeled data per task | Large-scale unsupervised/self-supervised pretraining on massive corpora, then fine-tuning or prompting for downstream tasks |
| Key Examples | TF-IDF classifiers, HMM POS taggers, CRF NER, SMT (statistical machine translation) | BERT, GPT, T5, RoBERTa, Claude |
| Advantages | Interpretable, lightweight, works with small datasets, low compute cost | Captures long-range context/semantics, strong generalization, transfer learning, handles ambiguity and multiple tasks with one model |
| Disadvantages | Poor generalization to unseen words/synonyms, brittle to context/ambiguity, heavy manual feature engineering | Requires huge data/compute to pretrain, less interpretable, risk of bias/hallucination, costly inference |

2) Discuss: How did the shift to modern NLP impact scalability and efficiency?

Modern NLP replaced hand-crafted, task-specific features with representation learning, so a single pretrained model can transfer to many downstream tasks instead of building a new pipeline for each one. Transformers' self-attention mechanism is fully parallelizable across tokens, unlike sequential RNNs, which lets training exploit GPU/TPU hardware far more efficiently and scale to billions of parameters and web-scale corpora. Transfer learning means most tasks only need fine-tuning or even zero/few-shot prompting on a pretrained model, drastically cutting the labeled data and compute needed per task compared to training traditional models from scratch. This shift traded a large upfront pretraining cost for much smaller marginal cost per downstream application, improving overall scalability even though individual models are far larger. It also enabled techniques like distillation and quantization (e.g., DistilBERT) to recover efficiency for deployment. Multi-task and multi-lingual generalization further reduced the need to maintain many separate specialized systems. Overall, scalability moved from "more manual engineering per task" to "more compute/data once, reused everywhere."


## 🌟 Exercise 2: LLM Architecture and Application Scenarios
For each, describe (a) core architectural differences, (b) a real application, (c) why it fits.

### BERT
- Bidirectional encoder-only Transformer: attends to both left and right context simultaneously; pretrained with Masked Language Modeling (predict randomly masked tokens) and historically Next Sentence Prediction. Produces rich contextual representations of input text rather than generating new text.
- Application: Sentiment classification / Named Entity Recognition / extractive Question Answering
- Why: These tasks need deep understanding of the full sentence (both preceding and following context) to correctly label or extract spans, but don't require generating new text — bidirectional encoding maximizes context available for each token's representation.

### GPT
- Decoder-only Transformer: autoregressive, causal (left-to-right) self-attention, so each token can only attend to previous tokens. Pretrained with a Causal Language Modeling objective (predict the next token).
- Application: Conversational chatbots, open-ended text generation, code completion/generation
- Why: These tasks require producing coherent new text token-by-token in sequence; the causal, generative structure of GPT is naturally suited to left-to-right text production, unlike BERT's bidirectional encoder which isn't built for generation.

### T5
- Encoder-decoder ("text-to-text") Transformer: an encoder reads the full input bidirectionally, then a decoder autoregressively generates output conditioned on the encoder's representation. Every task (classification, translation, summarization) is cast as text-in/text-out.
- Application: Summarization / machine translation
- Why: These tasks require both deeply understanding a full input sequence (favoring an encoder) and generating a transformed, variable-length output sequence (favoring a decoder) — the encoder-decoder split handles both needs, unlike encoder-only or decoder-only models.


## 🌟 Exercise 3: Benefits and Ethics of Pre-training
- Benefits (explain each in your words):
  1) Improved generalization: Pretraining on massive, diverse text exposes the model to broad language patterns, world knowledge, and syntax, so it performs better on new/unseen inputs rather than overfitting to one narrow dataset.
  2) Less labeled data: Because the model already learned general language structure during pretraining, downstream tasks only need a small labeled fine-tuning set (or none, with prompting) to reach strong performance.
  3) Faster fine-tuning: Starting from pretrained weights means training converges in far fewer steps than training a model from random initialization, since most of the useful representations are already learned.
  4) Transfer learning: The same pretrained backbone can be adapted to many different downstream tasks (classification, QA, summarization, etc.) instead of building a separate model from scratch for each one.
  5) Robustness: Exposure to huge, varied pretraining corpora makes models more resilient to noisy input, typos, rare words, and domain shift compared to models trained only on a small task-specific dataset.

- Ethical concerns: Pretrained LLMs can encode and amplify societal biases (gender, race, culture) present in training data; they can generate convincing misinformation or fabricated facts (hallucination); they can be misused for spam, phishing, or disinformation at scale; and training on scraped web data raises privacy concerns (memorization/leakage of personal information).
- Mitigations: Careful data curation and filtering to reduce toxic/biased content, differential privacy techniques during training to limit memorization of individual data points, safety filters and content moderation at inference time, RLHF (reinforcement learning from human feedback) to align outputs with human values, and regular bias/safety audits and red-teaming of deployed models.


## 🌟 Exercise 4: Transformer Architecture Deep Dive
### Self-Attention & Multi-Head Attention
- Each token's embedding is projected into three vectors: Query (Q), Key (K), and Value (V). For a given token, its Query is dot-producted with every token's Key to produce raw similarity scores; these scores are scaled and passed through softmax to produce attention weights that sum to 1. Those weights are then used to compute a weighted sum of all tokens' Value vectors, producing the output representation for that token — so each token's new representation is a context-aware blend of the whole sequence, weighted by relevance.
- Multiple heads help because each head learns its own Q/K/V projections into a different subspace, letting the model capture different types of relationships in parallel (e.g., one head might track syntactic dependencies, another might track coreference or semantic similarity). A single attention head would be forced to average all relation types into one weighting scheme, losing this diversity.
- Example sentence: "The trophy didn't fit in the suitcase because it was too big."
  - One head might learn coreference resolution, linking "it" strongly to "trophy" (or "suitcase" depending on context) to resolve the ambiguous pronoun.
  - Another head might capture syntactic structure, linking "fit" to its subject "trophy" and to the causal clause introduced by "because," capturing the verb-argument relation.

### Pre-training Objectives
- MLM vs CLM: MLM (Masked Language Modeling, used by BERT) randomly masks tokens in the input and trains the model to predict them using both left and right context — good for building bidirectional understanding. CLM (Causal Language Modeling, used by GPT) trains the model to predict the next token given only preceding tokens — good for autoregressive text generation.
- When to prefer MLM vs CLM: Prefer MLM-pretrained (encoder) models when the task is understanding/classification/extraction over a fixed input (NER, sentiment, QA span extraction). Prefer CLM-pretrained (decoder) models when the task requires generating new, open-ended text (chat, story generation, code completion).
- NSP: Early BERT used Next Sentence Prediction (predict whether sentence B follows sentence A) to try to teach inter-sentence coherence for tasks like QA. Many modern models (e.g., RoBERTa) dropped it because later research found NSP added little benefit and sometimes hurt performance — MLM alone, especially with better data/training, was sufficient and NSP's binary task was too easy/noisy to teach much useful signal.

### Transformer Model Selection
- Sentiment on reviews: Encoder-only (e.g., BERT/RoBERTa) — the task is classification over a fixed input requiring full bidirectional context, with no need to generate text.
- Conversational chatbot (creative responses): Decoder-only (e.g., GPT) — the task requires open-ended, autoregressive generation of novel text conditioned on conversation history.
- Technical document translation (EN→ES): Encoder-Decoder (e.g., T5/mT5, or a dedicated seq2seq translation model) — translation needs to fully understand the source sentence (encoder) and generate a fluent, structurally different target sentence (decoder).

### Positional Encoding
- Purpose: Self-attention itself is permutation-invariant — it has no inherent notion of token order, since it just computes weighted sums over all tokens' Q/K/V. Positional encodings inject information about each token's position (absolute or relative) into its embedding so the model can distinguish word order and structure, which is essential for meaning in language.
- Example failure without positions: Without positional encoding, "the dog bit the man" and "the man bit the dog" would produce identical sets of token representations under self-attention (same tokens, same weighted combination regardless of order), so the model couldn't distinguish who bit whom — losing critical meaning that depends purely on word order.


## 🌟 Exercise 5: BERT Variations: Choose Your Detective
Assign the best fit and justify:
- Scenario 1 (mobile, limited resources): DistilBERT — it's distilled to ~40% fewer parameters and ~60% faster than BERT while retaining ~97% of its performance, making it well suited to constrained mobile/edge deployment.
- Scenario 2 (legal docs, high accuracy): RoBERTa — trained longer on more data with better hyperparameters (dynamic masking, larger batches, no NSP), giving it stronger raw accuracy than base BERT, valuable for high-stakes legal text understanding.
- Scenario 3 (multilingual support): XLM-R — pretrained on massive multilingual corpora (CommonCrawl) across 100 languages, purpose-built for cross-lingual transfer and multilingual tasks.
- Scenario 4 (efficient pretraining with token replacement detection): ELECTRA — uses a replaced-token-detection objective (a small generator swaps tokens, a discriminator predicts which were replaced) instead of MLM, learning from every token rather than just masked ones, making pretraining much more sample-efficient.
- Scenario 5 (efficient NLP, constrained environments): ALBERT — uses parameter-sharing across layers and factorized embedding parameterization to drastically cut model size while retaining strong performance, ideal where memory is constrained.

Create/completed table:

| Model | Training differences | Size/Efficiency | Innovations | Ideal use cases |
|---|---|---|---|---|
| RoBERTa | Removes NSP, uses dynamic masking, trains longer with larger batches and more data | Same architecture/size as BERT but higher compute cost to train | Better-tuned pretraining recipe (data, batch size, masking) than original BERT | Tasks needing maximum accuracy where compute for training/fine-tuning isn't the bottleneck (e.g., legal/medical text classification) |
| ALBERT | Same MLM-style pretraining but with a sentence-order prediction (SOP) task instead of NSP | Much smaller memory footprint via cross-layer parameter sharing and factorized embeddings | Parameter sharing across transformer layers, embedding factorization | Memory-constrained environments needing BERT-level depth without BERT-level parameter count |
| DistilBERT | Trained via knowledge distillation from a BERT "teacher" model | ~40% smaller, ~60% faster inference, ~97% of BERT's performance retained | Knowledge distillation (student mimics teacher's output distribution/hidden states) | Mobile/edge deployment, latency-sensitive applications |
| ELECTRA | Replaced Token Detection: generator proposes replacement tokens, discriminator predicts real vs. replaced for every token | Comparable size to BERT but far more sample/compute-efficient pretraining | Discriminative pretraining objective using all tokens (not just ~15% masked) | Settings where pretraining compute/data budget is limited but strong downstream performance is needed |
| XLM-R | Pretrained on CommonCrawl data spanning ~100 languages, RoBERTa-style objective | Larger vocabulary/embedding table to cover many languages; larger overall model | Massively multilingual pretraining enabling strong cross-lingual transfer, even zero-shot | Multilingual or cross-lingual applications (e.g., global search, cross-language classification) |


## 🌟 Exercise 6: Softmax Temperature: The Randomness Regulator
### 1) Temperature Scenarios
- T=0.2: Output distribution is sharpened toward the most likely tokens — generation becomes very deterministic, repetitive, and conservative (low diversity, high coherence/predictability). Good for factual or precise tasks.
- T=1.5: Output distribution is flattened, giving lower-probability tokens much more chance of being picked — generation becomes more random, creative, and diverse, but at higher risk of incoherence or nonsensical output.
- T=1.0: The raw model distribution is used unmodified (no sharpening or flattening) — a balanced default between determinism and diversity.

### 2) Application Design
- Bedtime stories (creativity vs coherence): Use a moderate-to-high temperature (~0.8–1.2), possibly combined with top-p/top-k sampling, to allow creative, varied storytelling while still keeping sentences coherent — full determinism (low T) would make stories bland and repetitive.
- Financial report summaries (accuracy/reliability): Use a low temperature (~0.0–0.3) or greedy/beam decoding, since these outputs need to be precise, consistent, and factually reliable — creativity and randomness are undesirable and risk introducing incorrect figures or claims.

### 3) Temperature & Bias
- Higher temperature increases the chance of sampling rare, low-probability tokens, which can surface latent biased or edge-case associations buried in the tail of the distribution (since the model is more willing to explore unlikely completions) — for example, at high T a model might occasionally generate a stereotyped completion for an occupation-gender prompt that it would almost never produce at low T, where it defaults to the single most probable (often more neutral, high-frequency) token. Conversely, very low temperature can also dampen visible bias by always picking the "safest" statistically dominant completion, but this just hides bias rather than removing it — the biased associations still exist in the underlying probability distribution.


### (Optional) Quick Generation Demo Across Temperatures
Note: Requires `transformers` and model download; skip if offline.


In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

model_name = "gpt2"  # causal LM used for demo generation
prompt = "In the next decade, artificial intelligence will"

pipe = pipeline(
    "text-generation",
    model=AutoModelForCausalLM.from_pretrained(model_name),
    tokenizer=AutoTokenizer.from_pretrained(model_name),
    device=0 if torch.cuda.is_available() else -1,
)

for T in [0.2, 1.0, 1.5]:
    out = pipe(prompt, max_new_tokens=40, temperature=T, do_sample=True)
    print("--- Temperature:", T)
    print(out[0]["generated_text"])  # Inspect how style changes with T


/Users/jingjingzhao/Projects/DI-Bootcacmp/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-18 22:41:44.827628: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


--- Temperature: 0.2
In the next decade, artificial intelligence will become the dominant field in the field of artificial intelligence.

The next generation of artificial intelligence will be able to understand and understand human emotions and emotions in a way that is not possible with human speech


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


--- Temperature: 1.0
In the next decade, artificial intelligence will be used for a wide variety of jobs. But not just because of the technology itself, but because we need tools for tracking who lives in our data centers. We cannot trust technology without making use of
--- Temperature: 1.5
In the next decade, artificial intelligence will be the mainstream technology for the future of AI. However, those trends continue to show that their pace is far premature — from 2015 to 2025 an estimated 140 billion robots will be built worldwide in six years
